# Uncertainty Quantification Visualization Demo

This notebook demonstrates uncertainty quantification on CIFAR-10N with:
- **Small subset sampling** (configurable via parameters)
- **Multiple feature extractors**: DINOv2 (768-dim) or ResNet-10 (512-dim)
- **Decision boundary visualization** using dimensionality reduction
- **Aleatoric vs Epistemic uncertainty** distinction

## Key Concepts

### Feature Extractors
- **DINOv2**: Self-supervised vision transformer, 768-dim embeddings, slower but more powerful
- **ResNet-10**: Lightweight CNN, 512-dim embeddings, faster for quick experiments

### Architecture
- **Feature extraction**: Pre-trained backbone (DINOv2 or ResNet-10)
- **MLP classifier**: 2-layer fully connected network (input_dim → 256 → 10)
- **MC Dropout**: Enables uncertainty estimation via multiple stochastic forward passes

### Decision Boundaries
Since we work with high-dimensional embeddings (768D or 512D), we use **t-SNE or UMAP** to project to 2D for visualization.
The decision boundaries are visualized by:
1. Creating a 2D grid in the reduced space
2. Mapping grid points back to approximate high-D embeddings
3. Computing predictions and uncertainty for each grid point
4. Coloring regions by predicted class and uncertainty level

In [ ]:
# ============================================================================
# CONFIGURATION PARAMETERS - Adjust these to control the demo
# ============================================================================

# Feature extractor selection
FEATURE_EXTRACTOR = 'resnet10'  # 'dinov2' or 'resnet10'
#   - 'dinov2': 768-dim, slower, more powerful (requires transformers library)
#   - 'resnet10': 512-dim, faster, good for quick experiments

# Dataset sampling
SAMPLE_SIZE = 500  # Total samples to use (smaller = faster, larger = more representative)
TRAIN_RATIO = 0.7  # Fraction of samples for training
UNDER_SUPPORTED_CLASSES = [0, 1]  # Classes to intentionally under-support (epistemic uncertainty)
UNDER_TRAIN_PER_CLASS = 10  # Training samples per under-supported class
REGULAR_TRAIN_PER_CLASS = 50  # Training samples per regular class

# Model training
EPOCHS = 20  # Training epochs (more = better fit, but slower)
BATCH_SIZE = 32
LEARNING_RATE = 0.001
HIDDEN_DIM = 256  # MLP hidden layer size
DROPOUT = 0.2  # Dropout probability for MC Dropout

# Uncertainty estimation
MC_SAMPLES = 30  # Number of MC Dropout forward passes (more = more stable estimates)

# Visualization
REDUCTION_METHOD = 'umap'  # 'tsne' or 'umap' for dimensionality reduction
GRID_RESOLUTION = 100  # Resolution of decision boundary grid (higher = smoother but slower)
RANDOM_SEED = 42

print(f"Configuration loaded:")
print(f"  Feature extractor: {FEATURE_EXTRACTOR.upper()}")
print(f"  Sample size: {SAMPLE_SIZE}")
print(f"  Under-supported classes: {UNDER_SUPPORTED_CLASSES}")
print(f"  Training epochs: {EPOCHS}")
print(f"  MC samples: {MC_SAMPLES}")
print(f"  Reduction method: {REDUCTION_METHOD}")

In [ ]:
# Import required libraries
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from torchvision import models, transforms

# Dimensionality reduction
from sklearn.manifold import TSNE
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("⚠️ UMAP not available. Install with: pip install umap-learn")

# Add project paths
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / 'uq_classification'))

# Import project modules
from src.data.cifar10n_loader import CIFAR10NDataset
from uq_classification.data_loader import sample_indices_for_fast_pilot
from uq_classification.models import EmbeddingDataset, EmbeddingDropoutMLP

# Set random seeds
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Load CIFAR-10N Dataset

CIFAR-10N contains:
- **Clean labels**: Ground truth labels
- **Noisy labels**: Labels with human annotation noise (aleatoric uncertainty)
- **Noise mask**: Boolean mask indicating which samples have noisy labels

In [ ]:
# Load CIFAR-10N dataset
print("Loading CIFAR-10N dataset...")
dataset = CIFAR10NDataset(
    root='./cache/cifar10n',
    train=True,
    noise_type='worse_label',  # Use human annotation noise
    download=True
)

print(f"Dataset size: {len(dataset)}")
print(f"Number of noisy samples: {dataset.noise_mask.sum() if dataset.noise_mask is not None else 0}")
print(f"Noise rate: {dataset.noise_mask.sum() / len(dataset) * 100:.1f}%" if dataset.noise_mask is not None else "N/A")

# CIFAR-10 class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

## 2. Sample Train/Eval Splits

We create controlled splits to induce different types of uncertainty:
- **Under-supported classes**: Few training samples → **Epistemic uncertainty**
- **Noisy labels**: Label disagreement → **Aleatoric uncertainty**
- **Clean samples**: Well-supported with clean labels → **Low uncertainty**

In [ ]:
# Sample indices with controlled class support
print("\nSampling train/eval splits...")
split_spec = sample_indices_for_fast_pilot(
    dataset,
    under_supported_classes=UNDER_SUPPORTED_CLASSES,
    under_train_per_class=UNDER_TRAIN_PER_CLASS,
    regular_train_per_class=REGULAR_TRAIN_PER_CLASS,
    eval_per_group=50,  # 50 samples per evaluation group
    seed=RANDOM_SEED
)

print(f"\nSplit statistics:")
print(f"  Training samples: {len(split_spec.train_indices)}")
print(f"  Clean eval samples: {len(split_spec.clean_eval_indices)}")
print(f"  Aleatoric eval samples: {len(split_spec.aleatoric_eval_indices)}")
print(f"  Epistemic eval samples: {len(split_spec.epistemic_eval_indices)}")
print(f"  Under-supported classes: {[class_names[c] for c in split_spec.under_supported_classes]}")

## 3. Extract Features

We extract features using the selected backbone:
- **DINOv2**: 768-dimensional embeddings from self-supervised vision transformer
- **ResNet-10**: 512-dimensional embeddings from lightweight CNN

These embeddings capture semantic visual features and serve as input to our classifier.

In [ ]:
class ResNet10FeatureExtractor(nn.Module):
    """
    ResNet-10 feature extractor.
    Uses a lightweight ResNet architecture for fast feature extraction.
    """
    def __init__(self):
        super().__init__()
        # Use ResNet-18 as base (closest to ResNet-10 in torchvision)
        resnet = models.resnet18(pretrained=True)
        # Remove the final FC layer to get features
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.feature_dim = 512
        
    def forward(self, x):
        """
        Extract features from images.
        
        Args:
            x: Input images [batch_size, 3, 32, 32]
            
        Returns:
            features: Feature vectors [batch_size, 512]
        """
        features = self.features(x)
        return features.squeeze(-1).squeeze(-1)  # Remove spatial dimensions


def extract_features_with_backbone(dataset, indices, backbone_type='resnet10', batch_size=32, device='cpu'):
    """
    Extract features using specified backbone.
    
    Args:
        dataset: CIFAR-10N dataset
        indices: Indices to extract features for
        backbone_type: 'dinov2' or 'resnet10'
        batch_size: Batch size for extraction
        device: Device to run on
        
    Returns:
        Dictionary with features, labels, and metadata
    """
    subset = Subset(dataset, list(indices))
    loader = DataLoader(subset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    # Initialize backbone
    if backbone_type == 'dinov2':
        from src.models.dinov2_backbone import create_dinov2_model
        model = create_dinov2_model(
            model_name='dinov2_vits14',
            num_classes=10,
            dropout_rate=0.0,
            mc_dropout=False,
    features=train_features,
    labels=train_labels,
    clean_labels=train_clean_labels,
    is_noisy=train_is_noisy,
    original_indices=torch.arange(len(train_features))
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Initialize model
model = EmbeddingDropoutMLP(
    input_dim=768,
    num_classes=10,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"\nTraining MLP classifier...")
print(f"  Architecture: 768 → {HIDDEN_DIM} → 10")
print(f"  Dropout: {DROPOUT}")
print(f"  Epochs: {EPOCHS}")

# Training loop
train_losses = []
for epoch in tqdm(range(EPOCHS), desc="Training"):
    model.train()
    epoch_loss = 0.0
    
    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1}/{EPOCHS}, Loss: {avg_loss:.4f}")

print("\n✓ Training complete!")

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(train_losses, linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training Loss Curve', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Compute Uncertainty Estimates

We use **MC Dropout** to estimate uncertainty:
1. Enable dropout during inference
2. Perform multiple forward passes (MC samples)
3. Compute statistics from the prediction distribution:
   - **Mean prediction**: Average predicted class
   - **Predictive entropy**: Uncertainty in the prediction
   - **Mutual information**: Epistemic uncertainty (model uncertainty)
   - **Expected entropy**: Aleatoric uncertainty (data uncertainty)

In [ ]:
@torch.no_grad()
def mc_dropout_predictions(model, features, n_samples=30):
    """
    Perform MC Dropout inference to estimate uncertainty.
    
    Returns:
        mean_probs: Mean predicted probabilities [N, num_classes]
        predictive_entropy: Total uncertainty [N]
        mutual_information: Epistemic uncertainty [N]
        expected_entropy: Aleatoric uncertainty [N]
    """
    model.eval()
    model.enable_dropout()  # Keep dropout active
    
    features = features.to(device)
    all_probs = []
    
    for _ in range(n_samples):
        logits = model(features)
        probs = torch.softmax(logits, dim=1)
        all_probs.append(probs)
    
    all_probs = torch.stack(all_probs)  # [n_samples, N, num_classes]
    
    # Mean prediction
    mean_probs = all_probs.mean(dim=0)  # [N, num_classes]
    
    # Predictive entropy (total uncertainty)
    predictive_entropy = -(mean_probs * torch.log(mean_probs + 1e-10)).sum(dim=1)
    
    # Expected entropy (aleatoric uncertainty)
    entropies = -(all_probs * torch.log(all_probs + 1e-10)).sum(dim=2)  # [n_samples, N]
    expected_entropy = entropies.mean(dim=0)  # [N]
    
    # Mutual information (epistemic uncertainty)
    mutual_information = predictive_entropy - expected_entropy
    
    return mean_probs, predictive_entropy, mutual_information, expected_entropy

# Compute uncertainty for eval set
print(f"\nComputing uncertainty estimates with {MC_SAMPLES} MC samples...")
mean_probs, pred_entropy, mutual_info, expected_entropy = mc_dropout_predictions(
    model, eval_features, n_samples=MC_SAMPLES
)

# Get predicted classes
pred_classes = mean_probs.argmax(dim=1)

print("\n✓ Uncertainty computation complete!")
print(f"  Predictive entropy range: [{pred_entropy.min():.3f}, {pred_entropy.max():.3f}]")
print(f"  Mutual information range: [{mutual_info.min():.3f}, {mutual_info.max():.3f}]")
print(f"  Expected entropy range: [{expected_entropy.min():.3f}, {expected_entropy.max():.3f}]")

## 6. Dimensionality Reduction for Visualization

We reduce the 768-dimensional embeddings to 2D using t-SNE or UMAP.
This allows us to visualize:
- **Data distribution** in the embedding space
- **Decision boundaries** of the classifier
- **Uncertainty regions** where the model is uncertain

In [ ]:
# Combine train and eval features for consistent reduction
all_features_for_reduction = torch.cat([train_features, eval_features], dim=0).cpu().numpy()

print(f"\nPerforming {REDUCTION_METHOD.upper()} dimensionality reduction...")
print(f"  Input dimension: {all_features_for_reduction.shape[1]}")
print(f"  Output dimension: 2")

if REDUCTION_METHOD == 'tsne':
    reducer = TSNE(n_components=2, random_state=RANDOM_SEED, perplexity=30)
    reduced_features = reducer.fit_transform(all_features_for_reduction)
elif REDUCTION_METHOD == 'umap' and UMAP_AVAILABLE:
    reducer = umap.UMAP(n_components=2, random_state=RANDOM_SEED, n_neighbors=15)
    reduced_features = reducer.fit_transform(all_features_for_reduction)
else:
    print("⚠️ Falling back to t-SNE")
    reducer = TSNE(n_components=2, random_state=RANDOM_SEED, perplexity=30)
    reduced_features = reducer.fit_transform(all_features_for_reduction)

# Split back into train and eval
train_reduced = reduced_features[:len(train_features)]
eval_reduced = reduced_features[len(train_features):]

print("\n✓ Dimensionality reduction complete!")
print(f"  Reduced features shape: {reduced_features.shape}")

## 7. Visualize Data Distribution

First, let's visualize how the different uncertainty types are distributed in the 2D space.

In [ ]:
# Create visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Define colors for uncertainty types
uncertainty_colors = ['green', 'orange', 'red']
uncertainty_labels = ['Clean (Low Uncertainty)', 'Aleatoric (Noisy Labels)', 'Epistemic (Under-supported)']

# Plot 1: By uncertainty type
for i, (color, label) in enumerate(zip(uncertainty_colors, uncertainty_labels)):
    mask = eval_uncertainty_type == i
    axes[0].scatter(
        eval_reduced[mask, 0], eval_reduced[mask, 1],
        c=color, label=label, alpha=0.6, s=50, edgecolors='black', linewidth=0.5
    )
axes[0].set_title('Samples by Uncertainty Type', fontsize=14, fontweight='bold')
axes[0].legend(loc='best', fontsize=10)
axes[0].set_xlabel('Dimension 1', fontsize=12)
axes[0].set_ylabel('Dimension 2', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Plot 2: By true class
scatter = axes[1].scatter(
    eval_reduced[:, 0], eval_reduced[:, 1],
    c=eval_clean_labels.cpu().numpy(), cmap='tab10',
    alpha=0.6, s=50, edgecolors='black', linewidth=0.5
)
axes[1].set_title('Samples by True Class', fontsize=14, fontweight='bold')
cbar = plt.colorbar(scatter, ax=axes[1], ticks=range(10))
cbar.set_label('Class', fontsize=12)
axes[1].set_xlabel('Dimension 1', fontsize=12)
axes[1].set_ylabel('Dimension 2', fontsize=12)
axes[1].grid(True, alpha=0.3)

# Plot 3: By predictive entropy (total uncertainty)
scatter = axes[2].scatter(
    eval_reduced[:, 0], eval_reduced[:, 1],
    c=pred_entropy.cpu().numpy(), cmap='YlOrRd',
    alpha=0.6, s=50, edgecolors='black', linewidth=0.5
)
axes[2].set_title('Predictive Entropy (Total Uncertainty)', fontsize=14, fontweight='bold')
cbar = plt.colorbar(scatter, ax=axes[2])
cbar.set_label('Entropy', fontsize=12)
axes[2].set_xlabel('Dimension 1', fontsize=12)
axes[2].set_ylabel('Dimension 2', fontsize=12)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Decision Boundary Visualization

Now we visualize the decision boundaries by:
1. Creating a 2D grid in the reduced space
2. For each grid point, finding the nearest training sample in the reduced space
3. Using that sample's 768D embedding as an approximation
4. Computing predictions and uncertainty for the grid
5. Coloring regions by predicted class and uncertainty

**Note**: This is an approximation since we can't perfectly invert the dimensionality reduction.

In [ ]:
from scipy.spatial import cKDTree

# Create 2D grid
x_min, x_max = reduced_features[:, 0].min() - 1, reduced_features[:, 0].max() + 1
y_min, y_max = reduced_features[:, 1].min() - 1, reduced_features[:, 1].max() + 1
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, GRID_RESOLUTION),
    np.linspace(y_min, y_max, GRID_RESOLUTION)
)
grid_points = np.c_[xx.ravel(), yy.ravel()]

print(f"\nCreating decision boundary grid...")
print(f"  Grid resolution: {GRID_RESOLUTION}x{GRID_RESOLUTION}")
print(f"  Total grid points: {len(grid_points)}")

# Build KD-tree for nearest neighbor search
tree = cKDTree(reduced_features)

# Find nearest training sample for each grid point
print("  Finding nearest neighbors...")
distances, indices = tree.query(grid_points, k=1)

# Get corresponding 768D embeddings
grid_embeddings = all_features_for_reduction[indices]
grid_embeddings_tensor = torch.from_numpy(grid_embeddings).float()

# Compute predictions for grid
print("  Computing predictions for grid...")
with torch.no_grad():
    model.eval()
    grid_logits = model(grid_embeddings_tensor.to(device))
    grid_probs = torch.softmax(grid_logits, dim=1)
    grid_preds = grid_probs.argmax(dim=1).cpu().numpy()
    grid_confidence = grid_probs.max(dim=1)[0].cpu().numpy()

# Reshape for plotting
Z_class = grid_preds.reshape(xx.shape)
Z_confidence = grid_confidence.reshape(xx.shape)

print("\n✓ Decision boundary computation complete!")

In [ ]:
# Plot decision boundaries
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: Decision boundaries by class
contour = axes[0].contourf(xx, yy, Z_class, levels=np.arange(11) - 0.5, cmap='tab10', alpha=0.3)
axes[0].scatter(
    eval_reduced[:, 0], eval_reduced[:, 1],
    c=eval_clean_labels.cpu().numpy(), cmap='tab10',
    s=100, edgecolors='black', linewidth=1.5, alpha=0.8
)
axes[0].set_title('Decision Boundaries (Predicted Class)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Dimension 1', fontsize=12)
axes[0].set_ylabel('Dimension 2', fontsize=12)
cbar = plt.colorbar(contour, ax=axes[0], ticks=range(10))
cbar.set_label('Predicted Class', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Plot 2: Confidence map
contour = axes[1].contourf(xx, yy, Z_confidence, levels=20, cmap='RdYlGn', alpha=0.6)
# Overlay samples colored by uncertainty type
for i, (color, label) in enumerate(zip(uncertainty_colors, uncertainty_labels)):
    mask = eval_uncertainty_type == i
    axes[1].scatter(
        eval_reduced[mask, 0], eval_reduced[mask, 1],
        c=color, label=label, s=100, edgecolors='black', linewidth=1.5, alpha=0.8
    )
axes[1].set_title('Model Confidence Map', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Dimension 1', fontsize=12)
axes[1].set_ylabel('Dimension 2', fontsize=12)
axes[1].legend(loc='best', fontsize=10)
cbar = plt.colorbar(contour, ax=axes[1])
cbar.set_label('Confidence', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Uncertainty Decomposition Analysis

Compare the three types of uncertainty:
- **Predictive Entropy**: Total uncertainty
- **Mutual Information**: Epistemic uncertainty (model uncertainty)
- **Expected Entropy**: Aleatoric uncertainty (data uncertainty)

In [ ]:
# Prepare data for box plots
uncertainty_data = {
    'Predictive Entropy': pred_entropy.cpu().numpy(),
    'Mutual Information\n(Epistemic)': mutual_info.cpu().numpy(),
    'Expected Entropy\n(Aleatoric)': expected_entropy.cpu().numpy()
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Box plots for each uncertainty type
for idx, (metric_name, values) in enumerate(uncertainty_data.items()):
    data_by_type = [
        values[eval_uncertainty_type == 0],  # Clean
        values[eval_uncertainty_type == 1],  # Aleatoric
        values[eval_uncertainty_type == 2],  # Epistemic
    ]
    
    bp = axes[idx].boxplot(
        data_by_type,
        labels=['Clean', 'Aleatoric', 'Epistemic'],
        patch_artist=True,
        medianprops=dict(color='black', linewidth=2)
    )
    
    # Color boxes
    for patch, color in zip(bp['boxes'], uncertainty_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    
    axes[idx].set_title(metric_name, fontsize=14, fontweight='bold')
    axes[idx].set_ylabel('Value', fontsize=12)
    axes[idx].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Print statistics
print("\nUncertainty Statistics by Type:")
print("=" * 80)
for i, label in enumerate(['Clean', 'Aleatoric', 'Epistemic']):
    mask = eval_uncertainty_type == i
    print(f"\n{label}:")
    print(f"  Predictive Entropy: {pred_entropy[mask].mean():.3f} ± {pred_entropy[mask].std():.3f}")
    print(f"  Mutual Information:  {mutual_info[mask].mean():.3f} ± {mutual_info[mask].std():.3f}")
    print(f"  Expected Entropy:    {expected_entropy[mask].mean():.3f} ± {expected_entropy[mask].std():.3f}")

## 10. Classification Performance

Evaluate the classifier's accuracy on different uncertainty types.

In [ ]:
# Compute accuracy for each uncertainty type
correct = (pred_classes == eval_clean_labels).cpu().numpy()

print("\nClassification Accuracy by Uncertainty Type:")
print("=" * 60)
for i, label in enumerate(['Clean', 'Aleatoric', 'Epistemic']):
    mask = eval_uncertainty_type == i
    acc = correct[mask].mean() * 100
    print(f"{label:12s}: {acc:.1f}% ({correct[mask].sum()}/{mask.sum()})")

overall_acc = correct.mean() * 100
print(f"\nOverall Accuracy: {overall_acc:.1f}% ({correct.sum()}/{len(correct)})")

# Confusion matrix for epistemic samples (under-supported classes)
from sklearn.metrics import confusion_matrix
import seaborn as sns

epistemic_mask = eval_uncertainty_type == 2
cm = confusion_matrix(
    eval_clean_labels[epistemic_mask].cpu().numpy(),
    pred_classes[epistemic_mask].cpu().numpy(),
    labels=range(10)
)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names,
    cbar_kws={'label': 'Count'}
)
plt.title('Confusion Matrix (Epistemic/Under-supported Samples)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Class', fontsize=12)
plt.ylabel('True Class', fontsize=12)
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:

1. **Data Sampling**: Controlled train/eval splits to induce different uncertainty types
2. **Feature Extraction**: DINOv2 embeddings (768-dim) from CIFAR-10N images
3. **Model Training**: 2-layer MLP classifier with dropout
4. **Uncertainty Estimation**: MC Dropout for epistemic and aleatoric uncertainty
5. **Dimensionality Reduction**: t-SNE/UMAP for 2D visualization
6. **Decision Boundaries**: Approximate visualization in reduced space
7. **Uncertainty Analysis**: Decomposition into epistemic and aleatoric components

### Key Findings:

- **Clean samples** (well-supported, clean labels) → Low uncertainty, high accuracy
- **Aleatoric samples** (noisy labels) → High expected entropy, moderate accuracy
- **Epistemic samples** (under-supported classes) → High mutual information, lower accuracy

### Architecture Notes:

The MLP is a **fully connected network**, not convolutional:
- Input: 768-dim embeddings (pre-extracted features)
- Hidden: 256-dim with ReLU
- Output: 10-dim logits

Decision boundaries are **linear in the embedding space** but appear non-linear in the 2D projection due to:
1. The non-linear dimensionality reduction (t-SNE/UMAP)
2. The ReLU activation in the hidden layer
3. The high-dimensional nature of the original space (768D → 2D)

### Adjustable Parameters:

Modify the configuration cell at the top to:
- Change `SAMPLE_SIZE` for faster/slower experiments
- Adjust `UNDER_SUPPORTED_CLASSES` to test different epistemic scenarios
- Increase `MC_SAMPLES` for more stable uncertainty estimates
- Change `REDUCTION_METHOD` between 'tsne' and 'umap'
- Adjust `GRID_RESOLUTION` for smoother/faster boundary visualization